# 04 — Modeling: Logistic Regression + XGBoost (Session-level)

## Why modeling after A/B testing?
Our A/B analysis estimates the **average treatment effect** of UI variants on a high-intent conversion proxy.
Modeling complements this by:
- Identifying **key behavioral drivers** of conversion.
- Quantifying whether UI effects are **heterogeneous** (e.g., stronger for high-engagement sessions).
- Providing an interpretable baseline (Logistic Regression) and a stronger nonlinear model (XGBoost).

## Data
We use the session-level table derived from raw visit logs:
- `converted_hi`: label (1 if session has high-intent action, 0 otherwise)
- `group_mode`: UI variant assignment (categorical)
- session behavior features: event counts, duration, etc.

> Note: This is **not** a causal estimator. The experiment results remain the primary decision signal for shipping.


In [1]:
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
    classification_report,
    confusion_matrix,
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

# XGBoost
try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception as e:
    XGB_AVAILABLE = False
    print("XGBoost not available. Install with: pip install xgboost")
    print("Error:", e)

RANDOM_STATE = 42


## Load session-level dataset
We assume `02_ab_experiment.ipynb` saved `session_level.parquet` in the `data/` folder.

If you only have CSV, the code will fall back to CSV.


In [3]:
PATH_PARQUET = "../data/session_level.parquet"
PATH_CSV = "data/session_level.csv"

if os.path.exists(PATH_PARQUET):
    session = pd.read_parquet(PATH_PARQUET)
    source_used = PATH_PARQUET
elif os.path.exists(PATH_CSV):
    session = pd.read_csv(PATH_CSV)
    source_used = PATH_CSV
else:
    raise FileNotFoundError(
        "Cannot find session_level.parquet or session_level.csv in data/. "
        "Please save it from 02_ab_experiment.ipynb first."
    )

print("Loaded:", source_used)
print("Shape:", session.shape)
session.head(3)


Loaded: ../data/session_level.parquet
Shape: (100357, 10)


,session_start,session_end,n_events,n_unique_actions,group_mode,session_duration_sec,n_high_intent_events,converted_hi,is_treatment_any,is_bounce
session_token,,,,,,,,,,
1325679,2023-04-01 00:04:12,2023-04-01 00:04:12,1,0,default,0.0,0,0,0,1
1325691,2023-04-01 00:25:33,2023-04-01 00:25:33,1,0,default,0.0,0,0,0,1
1325701,2023-04-01 00:49:42,2023-04-01 00:49:42,1,0,default,0.0,0,0,0,1


## Basic checks & type fixes
We ensure:
- `converted_hi` is binary int
- duration is numeric seconds
- we have a clean `group_mode` string column


In [4]:
# Ensure label
if "converted_hi" not in session.columns:
    raise KeyError("Expected label column `converted_hi` not found in session.")

session["converted_hi"] = session["converted_hi"].astype(int)

# Ensure group_mode
if "group_mode" not in session.columns:
    # fallback: sometimes earlier code might name it differently
    if "visit_group" in session.columns:
        session["group_mode"] = session["visit_group"].astype(str)
    else:
        raise KeyError("Expected `group_mode` column not found (UI variant assignment).")

session["group_mode"] = session["group_mode"].astype(str)

# Ensure duration seconds numeric
if "session_duration_sec" in session.columns:
    # It might be timedelta; convert to float seconds if needed
    if np.issubdtype(session["session_duration_sec"].dtype, np.timedelta64):
        session["session_duration_sec"] = (
            session["session_duration_sec"].dt.total_seconds()
        )
    # if still not numeric, coerce
    session["session_duration_sec"] = pd.to_numeric(session["session_duration_sec"], errors="coerce")
else:
    # create if start/end exist
    if {"session_start", "session_end"}.issubset(set(session.columns)):
        session["session_start"] = pd.to_datetime(session["session_start"], errors="coerce")
        session["session_end"] = pd.to_datetime(session["session_end"], errors="coerce")
        session["session_duration_sec"] = (session["session_end"] - session["session_start"]).dt.total_seconds()
    else:
        session["session_duration_sec"] = np.nan

# Optional: ensure n_events exists
if "n_events" not in session.columns:
    # fallback: if you used other naming
    if "n_event" in session.columns:
        session["n_events"] = session["n_event"]
    else:
        session["n_events"] = np.nan

# Optional: n_unique_actions
if "n_unique_actions" not in session.columns:
    session["n_unique_actions"] = np.nan

session[["converted_hi", "group_mode", "session_duration_sec", "n_events", "n_unique_actions"]].head()


,converted_hi,group_mode,session_duration_sec,n_events,n_unique_actions
session_token,,,,,
1325679,0,default,0.0,1,0
1325691,0,default,0.0,1,0
1325701,0,default,0.0,1,0
1325705,0,default,0.0,1,0
1325706,0,default,0.0,1,0
